# Mass budget

A worked example of the `quicksat` mass budget against the sample satellite in `data/`: a small Earth observation platform with a telescope payload, hydrazine propulsion, and a separation interface split between the satellite and the launcher.

In [1]:
import os
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from sample/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat.mass.budget import MassBudget

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## Loading

The satellite is described by two files: a flat equipment CSV, and a config YAML holding the margins and harness fractions keyed on location.

In [2]:
DATA = Path("sample") / "data"
mass_data = MassBudget.from_csv(DATA / "equipment.csv", DATA / "budget_config.yaml")

f"Loaded mass data file and config: {len(mass_data.equipment)} rows, harness included"

'Loaded mass data file and config: 33 rows, harness included'

## The equipment table

Nothing is nested. `location` is the computation axis — it drives the system margin, the harness fraction, and what survives separation — while `responsibility` and `subsystem` exist so the same rows can be summed different ways.

Masses were parsed with pint on load, so the IMU's `750 g` has already become kilograms, and the harness rows at the bottom were derived rather than typed in.

In [3]:
mass_data.equipment[
    [
        "equipment_id",
        "location",
        "responsibility",
        "subsystem",
        "eqpt_mass",
        "number_of_units",
        "equipment_margin",
        "mass_class",
        "eqpt_total_mass",
    ]
]

,equipment_id,location,responsibility,subsystem,eqpt_mass,number_of_units,equipment_margin,mass_class,eqpt_total_mass
0,telescope,Payload,Payload,Instrument,62.00,1,20.00,equipment,62.00
1,focal_plane,Payload,Payload,Instrument,11.50,1,15.00,equipment,11.50
2,payload_electronics,Payload,Payload,OBDH,8.00,1,10.00,equipment,8.00
3,payload_radiator,Payload,Payload,Thermal,3.40,1,20.00,equipment,3.40
4,optical_bench,Payload,Payload,Structure,12.00,1,20.00,equipment,12.00
5,mli,Payload,Payload,Thermal,3.50,1,20.00,equipment,3.50
6,heaters,Payload,Payload,Thermal,1.20,1,15.00,equipment,1.20
7,star_tracker,Payload,Platform,ADCS,1.20,2,5.00,equipment,2.40
8,reaction_wheel,Platform,Platform,ADCS,7.00,4,10.00,equipment,28.00
9,magnetorquer,Platform,Platform,ADCS,1.10,3,10.00,equipment,3.30


## Asking for a mass

Every query takes the same four flags, all defaulting to `True`, so the usual question is a bare call and each deviation is one explicit switch: `wet` counts propellant, `sys_margin` applies the location's system margin, `eqpt_margin` applies the per-item margin, and `in_orbit` drops hardware left with the launcher.

The four masses normally quoted for a satellite are just two of those flags.

In [4]:
cases = {
    "launch mass (on ground, wet)": dict(in_orbit=False, wet=True),
    "dry mass at launch": dict(in_orbit=False, wet=False),
    "separated wet mass (in orbit)": dict(in_orbit=True, wet=True),
    "in-orbit dry mass": dict(in_orbit=True, wet=False),
}

pd.DataFrame(
    [
        {"case": label, "mass [kg]": mass_data.total_mass(**flags).to("kg").magnitude}
        for label, flags in cases.items()
    ]
).set_index("case")

,mass [kg]
case,
"launch mass (on ground, wet)",480.77
dry mass at launch,458.77
separated wet mass (in orbit),464.05
in-orbit dry mass,442.05


`in_orbit_mass()` and `on_ground_mass()` are presets over the same call, and the difference between them is exactly the launcher-side hardware: the adapter ring half and the clampband that stay behind at separation.

In [5]:
on_ground = mass_data.on_ground_mass()
in_orbit = mass_data.in_orbit_mass()

left_behind = mass_data.equipment.query("location == 'Launcher'")
print(f"on ground   {on_ground:~.2f}")
print(f"in orbit    {in_orbit:~.2f}")
print(f"difference  {(on_ground - in_orbit):~.2f}")
print()
print(left_behind[["equipment_id", "equipment_name", "eqpt_total_mass"]].to_string(index=False))

on ground   480.77 kg
in orbit    464.05 kg
difference  16.72 kg

equipment_id              equipment_name  eqpt_total_mass
 sep_ring_lv Adapter ring, launcher side            11.00
   clampband         Clampband and pyros             4.20


## The named queries

The rest of the API is the same call with a filter applied. Note that `platform_mass` and `payload_mass` take a `by_location` switch, because Platform and Payload are the names of both a location and a responsibility — the star tracker sits physically on the payload but belongs to the platform team, so the two axes disagree by exactly its margined mass.

`subsystem_mass` carries no system margin, since margins of that kind are held at the platform and payload level and cannot be attributed to a subsystem.

In [6]:
print(f"platform, by location        {mass_data.platform_mass():~.2f}")
print(f"payload,  by location        {mass_data.payload_mass():~.2f}")
print(f"payload,  by responsibility  {mass_data.payload_mass(by_location=False):~.2f}")
print()
print(f"ADCS, with margin            {mass_data.subsystem_mass('ADCS'):~.2f}")
print(f"ADCS, raw estimate           {mass_data.subsystem_mass('ADCS', eqpt_margin=False):~.2f}")
print(f"propulsion (wet)             {mass_data.subsystem_mass('Propulsion'):~.2f}")
print(f"propellant                   {mass_data.propellant_mass():~.2f}")

platform, by location        318.65 kg
payload,  by location        145.40 kg
payload,  by responsibility  142.50 kg

ADCS, with margin            39.50 kg
ADCS, raw estimate           36.05 kg
propulsion (wet)             33.66 kg
propellant                   22.00 kg


## Summing over the three axes

Each view is the same set of rows grouped a different way, so all three reconcile to the same total whatever the flags.

In [7]:
mass_data.by_subsystem()

,mass
subsystem,
ADCS,47.27
COMM,16.88
EPS,62.80
Harness,17.44
Instrument,100.77
OBDH,24.39
Propulsion,35.99
Structure,135.00
Thermal,23.50


In [8]:
mass_data.by_location()

,mass
location,
Payload,145.40
Platform,318.65


In [9]:
mass_data.by_responsibility()

,mass
responsibility,
Payload,142.50
Platform,321.55


Worth checking rather than assuming — the three groupings must agree in every case:

In [10]:
for label, flags in cases.items():
    total = mass_data.total_mass(**flags).to("kg").magnitude
    views = [
        mass_data.by_location(**flags)["mass"].sum(),
        mass_data.by_responsibility(**flags)["mass"].sum(),
        mass_data.by_subsystem(**flags)["mass"].sum(),
    ]
    agree = all(abs(view - total) < 1e-9 for view in views)
    print(f"{label:32s} {total:8.2f} kg   three views agree: {agree}")

launch mass (on ground, wet)       480.77 kg   three views agree: True
dry mass at launch                 458.77 kg   three views agree: True
separated wet mass (in orbit)      464.05 kg   three views agree: True
in-orbit dry mass                  442.05 kg   three views agree: True


## Where the margins land

Because the margin flags apply to the grouped views too, the three margin layers are the same call three times. Equipment mass before any margin, then with the per-item margin, then with the location's system margin on top.

In [11]:
pd.DataFrame(
    {
        "eqpt_total_mass": mass_data.by_subsystem(eqpt_margin=False, sys_margin=False)["mass"],
        "with_margin": mass_data.by_subsystem(sys_margin=False)["mass"],
        "with_sys_margin": mass_data.by_subsystem()["mass"],
    }
)

,eqpt_total_mass,with_margin,with_sys_margin
subsystem,,,
ADCS,36.05,39.50,47.27
COMM,12.70,14.06,16.88
EPS,45.30,52.34,62.80
Harness,13.34,14.68,17.44
Instrument,73.50,87.62,100.77
OBDH,18.90,20.70,24.39
Propulsion,32.20,33.66,35.99
Structure,95.00,113.10,135.00
Thermal,16.80,19.99,23.50


## Harness

Harness is never typed into the CSV. One row per location is derived as a percentage of that location's *equipment mass, before margin* — propellant is excluded, since cabling scales with the boxes it connects — and then carries its own contingency.

It takes its location's name as its responsibility, so it is counted whichever axis you sum on, while staying its own line in the subsystem view.

In [12]:
mass_data.equipment.query("subsystem == 'Harness'")[
    ["equipment_id", "location", "responsibility", "eqpt_total_mass", "equipment_margin", "comments"]
]

,equipment_id,location,responsibility,eqpt_total_mass,equipment_margin,comments
31,harness_payload,Payload,Payload,3.12,10.00,Derived: 3.0% of 104.000 kg equipment mass
32,harness_platform,Platform,Platform,10.22,10.00,Derived: 5.0% of 204.450 kg equipment mass


## The budget as a document

The scalar API answers single questions. `tabulated_mass()` produces the whole budget as a table: every item, grouped into subsystem blocks within each location, subtotalled, then the location subtotal before and after its system margin, and finally the dry mass, the propellant, and the wet mass.

Propellant appears once, at the bottom, and is left out of the blocks above it — so every subtotal on the way down is a dry mass and the column adds up as it reads.

Each row carries a `row_type`, so the frame can be styled, filtered or exported to a real budget document.

In [13]:
report = mass_data.tabulated_mass(in_orbit=True)

report[["label", "subsystem", "units", "eqpt_mass", "margin_pct", "eqpt_total_mass", "eqpt_total_mass_with_margin", "total_mass_with_sys_margin"]]

,label,subsystem,units,eqpt_mass,margin_pct,eqpt_total_mass,eqpt_total_mass_with_margin,total_mass_with_sys_margin
0,telescope,Instrument,1.00,62.00,20.00,62.00,74.40,NaN
1,focal_plane,Instrument,1.00,11.50,15.00,11.50,13.22,NaN
2,Instrument subtotal,Instrument,NaN,NaN,NaN,73.50,87.62,NaN
3,payload_electronics,OBDH,1.00,8.00,10.00,8.00,8.80,NaN
4,OBDH subtotal,OBDH,NaN,NaN,NaN,8.00,8.80,NaN
5,payload_radiator,Thermal,1.00,3.40,20.00,3.40,4.08,NaN
6,mli,Thermal,1.00,3.50,20.00,3.50,4.20,NaN
7,heaters,Thermal,1.00,1.20,15.00,1.20,1.38,NaN
8,Thermal subtotal,Thermal,NaN,NaN,NaN,8.10,9.66,NaN
9,optical_bench,Structure,1.00,12.00,20.00,12.00,14.40,NaN


The totals reconcile with the scalar API, which is the check that matters most here:

In [14]:
wet_total = report.loc[report["row_type"] == "wet_total", "total_mass_with_sys_margin"].iloc[0]
dry_total = report.loc[report["row_type"] == "dry_total", "total_mass_with_sys_margin"].iloc[0]

print(f"table wet total   {wet_total:8.2f} kg   vs in_orbit_mass()          {mass_data.in_orbit_mass().to('kg').magnitude:8.2f} kg")
print(f"table dry total   {dry_total:8.2f} kg   vs in_orbit_mass(wet=False) {mass_data.in_orbit_mass(wet=False).to('kg').magnitude:8.2f} kg")

table wet total     464.05 kg   vs in_orbit_mass()            464.05 kg
table dry total     442.05 kg   vs in_orbit_mass(wet=False)   442.05 kg
